# IMDb Sentiment Analysis
Loads training and test datasets into Delta tables.

In [0]:
# Install Hugging Face datasets + hub pinned to versions compatible with this
# runtime's built-in fsspec — the latest huggingface_hub causes a
# HfFileSystem "maxdepth" TypeError against the runtime's older fsspec.
%pip install datasets==2.20.0 huggingface_hub==0.24.6 --quiet

In [0]:
from datasets import load_dataset
from pyspark.sql.types import StructType, StructField, StringType, LongType

# 1. Open streaming connections (streaming=True)
# This does NOT download the data. It just opens pointers to the data.
hf_stream_train = load_dataset("stanfordnlp/imdb", split="train", streaming=True)
hf_stream_test = load_dataset("stanfordnlp/imdb", split="test", streaming=True)

# 2. Define the schema explicitly so Spark doesn't have to scan the data to infer it
schema = StructType([
    StructField("text", StringType(), True),
    StructField("label", LongType(), True)
])

# 3. Create a Python Generator to yield rows one-by-one or in tiny batches
def row_generator(hf_stream):
    for row in hf_stream:
        yield (row["text"], row["label"])

# 4. Turn the generator into a distributed Spark DataFrame
# Spark will pull from the generator lazily as needed, keeping memory flat.
spark_train_df = spark.createDataFrame(row_generator(hf_stream_train), schema=schema)
spark_test_df = spark.createDataFrame(row_generator(hf_stream_test), schema=schema)

# 5. Save it immediately to disk to let Spark process it in chunks
spark_train_df.write.mode("overwrite").saveAsTable("imdb_sentiment_train")
spark_test_df.write.mode("overwrite").saveAsTable("imdb_sentiment_test")

print("Successfully streamed and saved to Delta Tables")


In [0]:
%sql
SELECT * FROM imdb_sentiment_train LIMIT 5;

In [0]:
%sql
SELECT * FROM imdb_sentiment_test LIMIT 5;